# Agent 1 — File Reception playground

Demonstrates `FileReceptionAgent` end-to-end:
validation → SHA-256 → MIME detection → SSE events → audit record.

> **Kernel**: select `.venv` (Python 3.12) in the top-right kernel picker.

## 1 — Imports

In [1]:
import asyncio

from classiflow.ingesta.agents import FileReceptionAgent
from classiflow.ingesta.mime import detect_mime
from classiflow.shared.audit.service import AuditService
from classiflow.shared.database.repositories.audit import InMemoryAuditRepository
from classiflow.shared.events.broadcaster import EventBroadcaster

print("imports OK")

## 2 — Build the agent

`FileReceptionAgent` is a Pydantic model — pass dependencies as keyword arguments.
`detect_mime` is the production implementation (uses the `filetype` library).

In [2]:
audit_repo = InMemoryAuditRepository()
audit = AuditService(audit_repo)
broadcaster = EventBroadcaster()

agent = FileReceptionAgent(
    audit=audit,
    broadcaster=broadcaster,
    mime_detector=detect_mime,
)
print("agent ready")

## 3 — Run with a valid PDF

The cell uses top-level `await` — that works in Jupyter without `asyncio.run()`.

In [3]:
MINIMAL_PDF = (
    b"%PDF-1.4\n1 0 obj\n<< /Type /Catalog >>\nendobj\n"
    b"xref\n0 1\n0000000000 65535 f\ntrailer\n<< /Size 1 >>\nstartxref\n9\n%%EOF"
)

result = await agent.run(job_id="demo-001", filename="sample.pdf", file_bytes=MINIMAL_PDF)

print("=== File state ===")
print(f"  passed          : {result.passed}")
print(f"  sha256          : {result.sha256}")
print(f"  detected_mime   : {result.detected_mime}")
print(f"  file_size_bytes : {result.file_size_bytes}")
print(f"  rejection_reason: {result.rejection_reason}")

2026-06-21 18:16:47.791 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-001 agent=agent1_file_reception event=passed


## 4 — Inspect the audit record

In [5]:
records = await audit_repo.list_for_job("demo-001")

print("=== Audit records ===")
for r in records:
    print(f"  event       : {r.event}")
    print(f"  agent       : {r.agent}")
    print(f"  duration_ms : {r.duration_ms} ms")
    print(f"  detail      : {r.detail}")
    print()

## 5 — Observe SSE events in real time

The agent emits `STARTED` then `PASSED`/`FAILED` through the `EventBroadcaster`.
Here we subscribe before calling `run()` so we catch both events.

In [6]:
broadcaster2 = EventBroadcaster()
agent2 = FileReceptionAgent(
    audit=AuditService(InMemoryAuditRepository()),
    broadcaster=broadcaster2,
    mime_detector=detect_mime,
)

events = []


async def collect() -> None:
    async for event in broadcaster2.subscribe("demo-002"):
        events.append(event)
        print(f"  SSE → agent={event.agent}  status={event.status}")


collect_task = asyncio.create_task(collect())
await asyncio.sleep(0)

await agent2.run(job_id="demo-002", filename="sample.pdf", file_bytes=MINIMAL_PDF)
await broadcaster2.close("demo-002")
await collect_task

print(f"\ncollected {len(events)} events")

2026-06-21 18:17:04.810 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-002 agent=agent1_file_reception event=passed


## 6 — Rejection cases

Agent rejects: missing file, empty bytes, and anything over 50 MB.

In [7]:
cases = [
    ("no file", None),
    ("empty file", b""),
    ("oversized", b"x" * (51 * 1024 * 1024)),
]

print("=== Rejection cases ===")
for label, data in cases:
    r = await FileReceptionAgent(
        audit=AuditService(InMemoryAuditRepository()),
        broadcaster=EventBroadcaster(),
        mime_detector=detect_mime,
    ).run(job_id=f"demo-{label}", filename="test.pdf", file_bytes=data)
    print(f"  {label:12} → passed={r.passed}  reason='{r.rejection_reason}'")

2026-06-21 18:17:10.033 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-no file agent=agent1_file_reception event=failed
2026-06-21 18:17:10.034 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-empty file agent=agent1_file_reception event=failed
2026-06-21 18:17:10.034 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-oversized agent=agent1_file_reception event=failed


## 7 — Run with a real file from disk

Drop any PDF, DOCX, or image into:
```
src/classiflow/playground/samples/
```
then set `FILE_NAME` below and run the cell.

In [8]:
from pathlib import Path

# resolve samples/ relative to this notebook, regardless of where Jupyter was launched
samples_dir = next(
    p / "samples"
    for p in [Path.cwd(), Path.cwd() / "src" / "classiflow" / "playground"]
    if (p / "samples").is_dir()
)

files = sorted(samples_dir.iterdir())
if not files:
    msg = f"No files found in {samples_dir}. Drop a PDF/DOCX/image there first."
    raise FileNotFoundError(msg)

for _f in files:
    pass

file_path = files[0]
file_bytes = file_path.read_bytes()

real_result = await FileReceptionAgent(
    audit=AuditService(InMemoryAuditRepository()),
    broadcaster=EventBroadcaster(),
    mime_detector=detect_mime,
).run(job_id="demo-real", filename=file_path.name, file_bytes=file_bytes)

2026-06-21 18:17:26.224 | INFO     | classiflow.shared.audit.service:record:37 - audit | job=demo-real agent=agent1_file_reception event=passed


Found 1 file(s) in /Users/leonardoheis/Source/repo/Trabajo-Integrador/src/classiflow/playground/samples:
  decreto_ordenanza_818_1976.pdf

using: decreto_ordenanza_818_1976.pdf  (157,372 bytes)
passed=True sha256='f57c532180dad7463d1aef931582725efa269829ed0d30c2ce8d5b582b0fd4c7' detected_mime='application/pdf' file_size_bytes=157372 rejection_reason=''
